# Oracle Imitation Position Cloning

This notebook trains a dense desired-position policy from oracle trade labels.

Workflow:
- build oracle entry/exit trade pairs
- convert sparse oracle long and short trades into dense daily desired-position labels
- if a trade is active from entry to exit, days in that window are labeled `long` or `short`
- train a 3-state position classifier (`flat / long / short`) on all data before `2020-01-01`
- evaluate out of sample on `2020-01-01+`
- derive `buy / sell / short / cover / hold` from predicted position changes for backtesting

Notes:
- this uses dense desired-position targets instead of sparse event labels
- opposite-side flips go through `flat` first, so the rollout uses at most one action per day


In [1]:
import os

import django
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from django.apps import apps
from IPython.display import display

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
if not apps.ready:
    django.setup()

from analysis.context_family_classifier_backtest import build_context_family_daily_state_panel
from analysis.oracle_entry_exit_dataset import build_oracle_entry_exit_frames
from data.historical_prices import load_adjusted_price_frames
from domain.labels.specs import LabelBuildSpec
from ml.rl import (
    backtest_position_policy_equal_weight,
    build_expert_position_panel,
    build_transition_sample_weights,
    build_transition_training_mask,
    rollout_position_policy,
    train_position_cloning_hgb,
    train_position_cloning_rf,
)
from pipeline.universe_selection import DEFAULT_US_EXCHANGES, resolve_symbol_universe
from workflows.labels import build_oracle_labels

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
APP_CFG = {
    "model_family": "hgb",
    "dates": {
        "start_date": "1900-01-01",
        "train_cutoff": "2020-01-01",
        "backtest_start": "2025-01-01",
        "end_date": None,
    },
    "universe": {
        "country": "US",
        "exchanges": list(DEFAULT_US_EXCHANGES),
        "min_market_cap": 100_000_000_000.0,
        "exclude_pooled_vehicles": True,
        "size": None,
    },
    "labels": {
        "k_params": {"YE": [1, 2, 4, 8, 16]},
        "min_profit_pct": 0.10,
        "buy_execution": "adj_high",
        "sell_execution": "adj_low",
        "short_execution": "adj_low",
        "cover_execution": "adj_high",
        "trade_dedup_mode": "exact",
        "download_missing_prices": True,
    },
    "rf": {
        "n_estimators": 150,
        "random_state": 1337,
        "n_jobs": -1,
        "max_depth": 12,
        "max_features": "sqrt",
        "min_samples_leaf": 5,
        "min_samples_split": 2,
        "class_weight": "balanced",
        "max_samples": 0.35,
    },
    "hgb": {
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "max_depth": 6,
        "min_samples_leaf": 50,
        "l2_regularization": 0.0,
        "random_state": 1337,
        "early_stopping": False,
    },
    "sample_weighting": {
        "enabled": True,
        "near_window": 3,
        "transition_weight": 1.0,
        "near_weight": 0.5,
        "interior_weight": 0.1,
    },
    "train_sampling": {
        "enabled": False,
        "transition_keep_prob": 1.0,
        "near_keep_prob": 0.35,
        "interior_keep_prob": 0.05,
        "random_state": 1337,
    },
    "backtest": {
        "initial_balance": 100_000.0,
        "fee_bps": 5.0,
        "slippage_bps": 5.0,
    },
}

TRAIN_CUTOFF = pd.Timestamp(APP_CFG["dates"]["train_cutoff"])
BACKTEST_START = pd.Timestamp(APP_CFG["dates"]["backtest_start"])
APP_CFG


{'model_family': 'hgb',
 'dates': {'start_date': '1900-01-01',
  'train_cutoff': '2020-01-01',
  'backtest_start': '2025-01-01',
  'end_date': None},
 'universe': {'country': 'US',
  'exchanges': ['NASDAQ', 'NYSE', 'AMEX'],
  'min_market_cap': 100000000000.0,
  'exclude_pooled_vehicles': True,
  'size': None},
 'labels': {'k_params': {'YE': [1, 2, 4, 8, 16]},
  'min_profit_pct': 0.1,
  'buy_execution': 'adj_high',
  'sell_execution': 'adj_low',
  'short_execution': 'adj_low',
  'cover_execution': 'adj_high',
  'trade_dedup_mode': 'exact',
  'download_missing_prices': True},
 'rf': {'n_estimators': 150,
  'random_state': 1337,
  'n_jobs': -1,
  'max_depth': 12,
  'max_features': 'sqrt',
  'min_samples_leaf': 5,
  'min_samples_split': 2,
  'class_weight': 'balanced',
  'max_samples': 0.35},
 'hgb': {'learning_rate': 0.05,
  'max_iter': 200,
  'max_leaf_nodes': 31,
  'max_depth': 6,
  'min_samples_leaf': 50,
  'l2_regularization': 0.0,
  'random_state': 1337,
  'early_stopping': False},
 

In [3]:
universe = tuple(
    resolve_symbol_universe(
        min_market_cap=float(APP_CFG["universe"]["min_market_cap"]),
        country=str(APP_CFG["universe"]["country"]),
        exchanges=list(APP_CFG["universe"]["exchanges"]),
        exclude_pooled_vehicles=bool(APP_CFG["universe"]["exclude_pooled_vehicles"]),
        limit=APP_CFG["universe"]["size"],
    )
)
if not universe:
    raise ValueError("Universe screen returned no symbols.")

symbol_order = {symbol: idx for idx, symbol in enumerate(universe)}
universe_rows = list(
    __import__("fmp.models", fromlist=["Symbol"]).Symbol.objects.filter(symbol__in=universe)
    .values("symbol", "company_name", "sector", "industry", "exchange", "country", "market_cap", "payload")
)
universe_df = pd.DataFrame(universe_rows)
if not universe_df.empty:
    universe_df["symbol"] = universe_df["symbol"].astype(str).str.strip().str.upper()
    universe_df["sector"] = universe_df["sector"].fillna("").astype(str).str.strip().replace("", "Unknown")
    universe_df["industry"] = universe_df["industry"].fillna("").astype(str).str.strip().replace("", "Unknown")
    universe_df["sort_order"] = universe_df["symbol"].map(symbol_order)
    universe_df = universe_df.sort_values(["sort_order", "symbol"]).drop(columns=["sort_order", "payload"])

price_frames = load_adjusted_price_frames(
    list(universe),
    start_date=APP_CFG["dates"]["start_date"],
    end_date=APP_CFG["dates"]["end_date"],
)

price_lookup_rows = []
for symbol, frame in price_frames.items():
    if frame.empty:
        continue
    working = frame.reset_index().copy()
    working["symbol"] = symbol
    working["date_text"] = pd.to_datetime(working["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    for row in working.to_dict(orient="records"):
        price_lookup_rows.append(
            {
                "symbol": str(row.get("symbol") or "").strip().upper(),
                "date_text": str(row.get("date_text") or ""),
                "adj_open": row.get("adj_open"),
                "adj_high": row.get("adj_high"),
                "adj_low": row.get("adj_low"),
                "adj_close": row.get("adj_close"),
                "volume": row.get("volume"),
            }
        )
price_lookup_df = pd.DataFrame(price_lookup_rows)
for col in ["adj_open", "adj_high", "adj_low", "adj_close", "volume"]:
    if col in price_lookup_df.columns:
        price_lookup_df[col] = pd.to_numeric(price_lookup_df[col], errors="coerce")
price_lookup_df = price_lookup_df.drop_duplicates(subset=["symbol", "date_text"], keep="last").reset_index(drop=True)

label_spec = LabelBuildSpec(
    k_params=dict(APP_CFG["labels"]["k_params"]),
    min_profit_pct=float(APP_CFG["labels"]["min_profit_pct"]),
    buy_execution=str(APP_CFG["labels"]["buy_execution"]),
    sell_execution=str(APP_CFG["labels"]["sell_execution"]),
    short_execution=str(APP_CFG["labels"]["short_execution"]),
    cover_execution=str(APP_CFG["labels"]["cover_execution"]),
    trade_dedup_mode=str(APP_CFG["labels"]["trade_dedup_mode"]),
    start_date=APP_CFG["dates"]["start_date"],
    end_date=APP_CFG["dates"]["end_date"],
    download_missing_prices=bool(APP_CFG["labels"]["download_missing_prices"]),
)
oracle_result = build_oracle_labels(list(universe), spec=label_spec, price_frames=price_frames)
label_df = pd.DataFrame(oracle_result.label_rows)
completed_trades_df = pd.DataFrame(oracle_result.completed_trades)
trade_pair_df, state_df = build_oracle_entry_exit_frames(
    universe_df=universe_df,
    label_df=label_df,
    completed_trades_df=completed_trades_df,
    price_lookup_df=price_lookup_df,
)

run_config_df = pd.DataFrame([
    {
        "min_market_cap": float(APP_CFG["universe"]["min_market_cap"]),
        "k_params": str(APP_CFG["labels"]["k_params"]),
        "min_profit_pct": float(APP_CFG["labels"]["min_profit_pct"]),
        "buy_execution": str(APP_CFG["labels"]["buy_execution"]),
        "sell_execution": str(APP_CFG["labels"]["sell_execution"]),
        "short_execution": str(APP_CFG["labels"]["short_execution"]),
        "cover_execution": str(APP_CFG["labels"]["cover_execution"]),
        "train_cutoff": TRAIN_CUTOFF.date().isoformat(),
        "backtest_start": BACKTEST_START.date().isoformat(),
        "resolved_symbols": int(len(universe)),
        "oracle_trade_pairs": int(len(trade_pair_df)),
    }
])
display(run_config_df)


,min_market_cap,k_params,min_profit_pct,buy_execution,sell_execution,short_execution,cover_execution,train_cutoff,backtest_start,resolved_symbols,oracle_trade_pairs
0,1.000000e+11,"{'YE': [1, 2, 4, 8, 16]}",0.1,adj_high,adj_low,adj_low,adj_high,2020-01-01,2025-01-01,114,43166


In [ ]:
daily_state_df, technical_feature_cols, fundamental_feature_cols, macro_feature_cols = build_context_family_daily_state_panel(
    universe_df,
    price_frames,
    start_date=APP_CFG["dates"]["start_date"],
    end_date=APP_CFG["dates"]["end_date"],
)
STATE_MODEL_NUMERIC_COLS = technical_feature_cols + fundamental_feature_cols + macro_feature_cols
for col in STATE_MODEL_NUMERIC_COLS:
    if col in daily_state_df.columns:
        daily_state_df[col] = pd.to_numeric(daily_state_df[col], errors="coerce").fillna(0.0)

expert_panel_df = build_expert_position_panel(daily_state_df, trade_pair_df)
if APP_CFG["sample_weighting"]["enabled"]:
    expert_panel_df = build_transition_sample_weights(
        expert_panel_df,
        near_window=int(APP_CFG["sample_weighting"]["near_window"]),
        transition_weight=float(APP_CFG["sample_weighting"]["transition_weight"]),
        near_weight=float(APP_CFG["sample_weighting"]["near_weight"]),
        interior_weight=float(APP_CFG["sample_weighting"]["interior_weight"]),
    )
else:
    expert_panel_df["transition_distance_steps"] = np.nan
    expert_panel_df["sample_weight"] = 1.0

if APP_CFG["train_sampling"]["enabled"]:
    expert_panel_df = build_transition_training_mask(
        expert_panel_df,
        near_window=int(APP_CFG["sample_weighting"]["near_window"]),
        transition_keep_prob=float(APP_CFG["train_sampling"]["transition_keep_prob"]),
        near_keep_prob=float(APP_CFG["train_sampling"]["near_keep_prob"]),
        interior_keep_prob=float(APP_CFG["train_sampling"]["interior_keep_prob"]),
        random_state=int(APP_CFG["train_sampling"]["random_state"]),
    )
else:
    expert_panel_df["keep_for_training"] = True
    expert_panel_df["train_keep_probability"] = 1.0

train_mask = pd.to_datetime(expert_panel_df["date"], errors="coerce") < TRAIN_CUTOFF
expert_split_summary_df = pd.DataFrame([
    {
        "split": "train",
        "rows": int(train_mask.sum()),
        "fit_rows": int(expert_panel_df.loc[train_mask, "keep_for_training"].sum()),
        "symbols": int(expert_panel_df.loc[train_mask, "symbol"].nunique()),
        "long_rate": float((expert_panel_df.loc[train_mask, "expert_position_label"] == "long").mean()),
        "short_rate": float((expert_panel_df.loc[train_mask, "expert_position_label"] == "short").mean()),
        "flat_rate": float((expert_panel_df.loc[train_mask, "expert_position_label"] == "flat").mean()),
        "entry_signal_rate": float(expert_panel_df.loc[train_mask, "entry_signal"].mean()),
        "exit_signal_rate": float(expert_panel_df.loc[train_mask, "exit_signal"].mean()),
        "mean_sample_weight": float(expert_panel_df.loc[train_mask, "sample_weight"].mean()),
    },
    {
        "split": "out_of_sample",
        "rows": int((~train_mask).sum()),
        "fit_rows": int((~train_mask).sum()),
        "symbols": int(expert_panel_df.loc[~train_mask, "symbol"].nunique()),
        "long_rate": float((expert_panel_df.loc[~train_mask, "expert_position_label"] == "long").mean()),
        "short_rate": float((expert_panel_df.loc[~train_mask, "expert_position_label"] == "short").mean()),
        "flat_rate": float((expert_panel_df.loc[~train_mask, "expert_position_label"] == "flat").mean()),
        "entry_signal_rate": float(expert_panel_df.loc[~train_mask, "entry_signal"].mean()),
        "exit_signal_rate": float(expert_panel_df.loc[~train_mask, "exit_signal"].mean()),
        "mean_sample_weight": float(expert_panel_df.loc[~train_mask, "sample_weight"].mean()),
    },
])
display(expert_split_summary_df)
display(expert_panel_df[[col for col in ["symbol", "date_text", "expert_action", "expert_position_label", "transition_distance_steps", "sample_weight", "keep_for_training", "train_keep_probability"] if col in expert_panel_df.columns]].head(40))
print("Feature count:", len(STATE_MODEL_NUMERIC_COLS))


In [ ]:
if APP_CFG["model_family"] == "rf":
    position_result = train_position_cloning_rf(
        expert_panel_df,
        STATE_MODEL_NUMERIC_COLS,
        train_cutoff=TRAIN_CUTOFF,
        rf_kwargs=dict(APP_CFG["rf"]),
        sample_weight_col="sample_weight" if APP_CFG["sample_weighting"]["enabled"] else None,
        train_keep_col="keep_for_training" if APP_CFG["train_sampling"]["enabled"] else None,
    )
elif APP_CFG["model_family"] == "hgb":
    position_result = train_position_cloning_hgb(
        expert_panel_df,
        STATE_MODEL_NUMERIC_COLS,
        train_cutoff=TRAIN_CUTOFF,
        hgb_kwargs=dict(APP_CFG["hgb"]),
        sample_weight_col="sample_weight" if APP_CFG["sample_weighting"]["enabled"] else None,
        train_keep_col="keep_for_training" if APP_CFG["train_sampling"]["enabled"] else None,
    )
else:
    raise ValueError(f"Unsupported model_family: {APP_CFG['model_family']}")

behavior_cloning_summary_df = position_result.summary_df.copy()
behavior_cloning_summary_df.insert(0, "model_family", APP_CFG["model_family"])
behavior_cloning_report_df = position_result.report_df.copy()
behavior_cloning_feature_importance_df = position_result.feature_importance_df.copy()

preview_cols = [
    col for col in [
        "symbol",
        "date_text",
        "expert_position_label",
        "pred_position_label",
        "prob_flat",
        "prob_long",
        "prob_short",
        "entry_signal",
        "exit_signal",
        "expert_action",
        "transition_distance_steps",
        "sample_weight",
        "keep_for_training",
    ] if col in position_result.scored_oos_df.columns
]
behavior_cloning_preview_df = position_result.scored_oos_df[preview_cols].head(60).copy()

display(behavior_cloning_summary_df)
display(behavior_cloning_report_df)
display(behavior_cloning_feature_importance_df.head(40))
display(behavior_cloning_preview_df)


In [ ]:
def summarize_backtest(equity_s, return_s):
    total_return_pct = float((equity_s.iloc[-1] / equity_s.iloc[0] - 1.0) * 100.0) if len(equity_s) else float("nan")
    sharpe = float((return_s.mean() / return_s.std(ddof=0)) * np.sqrt(252.0)) if return_s.std(ddof=0) > 1e-12 else float("nan")
    max_drawdown_pct = float((((equity_s / equity_s.cummax()) - 1.0).min()) * 100.0) if len(equity_s) else float("nan")
    return total_return_pct, sharpe, max_drawdown_pct


def summarize_symbol_outperformance(stateful_df, *, policy_name, mode_name):
    per_symbol_df = (
        stateful_df.groupby("symbol", as_index=False)
        .agg(
            strategy_total_return=("strategy_return", lambda s: float(np.prod(1.0 + pd.Series(s).fillna(0.0)) - 1.0)),
            buy_hold_total_return=("next_return", lambda s: float(np.prod(1.0 + pd.Series(s).fillna(0.0)) - 1.0)),
        )
    )
    per_symbol_df["outperformed_buy_hold"] = per_symbol_df["strategy_total_return"] > per_symbol_df["buy_hold_total_return"]
    per_symbol_df["mode"] = mode_name
    per_symbol_df["policy"] = policy_name
    summary_df = pd.DataFrame([
        {
            "mode": mode_name,
            "policy": policy_name,
            "beat_symbols": int(per_symbol_df["outperformed_buy_hold"].sum()),
            "total_symbols": int(len(per_symbol_df)),
            "beat_rate": float(per_symbol_df["outperformed_buy_hold"].mean()) if len(per_symbol_df) else float("nan"),
            "mean_strategy_symbol_return_pct": float(per_symbol_df["strategy_total_return"].mean() * 100.0) if len(per_symbol_df) else float("nan"),
            "mean_buy_hold_symbol_return_pct": float(per_symbol_df["buy_hold_total_return"].mean() * 100.0) if len(per_symbol_df) else float("nan"),
        }
    ])
    return summary_df, per_symbol_df


def apply_position_mode(frame, *, source_col, output_col, mode):
    working = frame.copy()
    labels = working[source_col].fillna("flat").astype(str).str.lower()
    if mode == "long_only":
        working[output_col] = np.where(labels == "long", "long", "flat")
    elif mode == "short_only":
        working[output_col] = np.where(labels == "short", "short", "flat")
    elif mode == "long_short":
        working[output_col] = np.where(labels.isin(["long", "short"]), labels, "flat")
    else:
        raise ValueError(f"Unsupported mode: {mode}")
    return working


label_recipe_summary_df = pd.DataFrame([
    {
        "min_market_cap": float(APP_CFG["universe"]["min_market_cap"]),
        "k_params": str(APP_CFG["labels"]["k_params"]),
        "min_profit_pct": float(APP_CFG["labels"]["min_profit_pct"]),
        "buy_execution": APP_CFG["labels"]["buy_execution"],
        "sell_execution": APP_CFG["labels"]["sell_execution"],
        "short_execution": APP_CFG["labels"]["short_execution"],
        "cover_execution": APP_CFG["labels"]["cover_execution"],
        "train_cutoff": str(APP_CFG["dates"]["train_cutoff"]),
        "backtest_start": str(APP_CFG["dates"]["backtest_start"]),
        "sample_weighting": bool(APP_CFG["sample_weighting"]["enabled"]),
        "near_window": int(APP_CFG["sample_weighting"]["near_window"]),
        "transition_weight": float(APP_CFG["sample_weighting"]["transition_weight"]),
        "near_weight": float(APP_CFG["sample_weighting"]["near_weight"]),
        "interior_weight": float(APP_CFG["sample_weighting"]["interior_weight"]),
        "train_sampling": bool(APP_CFG["train_sampling"]["enabled"]),
    }
])
display(label_recipe_summary_df)

backtest_state_df = daily_state_df[pd.to_datetime(daily_state_df["date"], errors="coerce") >= BACKTEST_START].copy()
rollout_df = rollout_position_policy(
    backtest_state_df,
    STATE_MODEL_NUMERIC_COLS,
    model=position_result.model,
)
expert_backtest_df = expert_panel_df[pd.to_datetime(expert_panel_df["date"], errors="coerce") >= BACKTEST_START].copy()
buy_hold_backtest_df = backtest_state_df.copy()
buy_hold_backtest_df["policy_position_label"] = "long"

mode_results = []
mode_stateful_frames = {}
mode_equity_series = {}
beat_summary_frames = []
beat_detail_frames = []
for mode_name in ["long_only", "short_only", "long_short"]:
    model_mode_df = apply_position_mode(
        rollout_df.assign(policy_position_label=rollout_df["sim_position_label_after"]),
        source_col="policy_position_label",
        output_col="mode_position_label",
        mode=mode_name,
    )
    expert_mode_df = apply_position_mode(
        expert_backtest_df,
        source_col="expert_position_label",
        output_col="mode_position_label",
        mode=mode_name,
    )

    model_stateful_df, model_equity, model_returns = backtest_position_policy_equal_weight(
        model_mode_df,
        position_col="mode_position_label",
        initial_balance=float(APP_CFG["backtest"]["initial_balance"]),
        fee_bps=float(APP_CFG["backtest"]["fee_bps"]),
        slippage_bps=float(APP_CFG["backtest"]["slippage_bps"]),
    )
    expert_stateful_df, expert_equity, expert_returns = backtest_position_policy_equal_weight(
        expert_mode_df,
        position_col="mode_position_label",
        initial_balance=float(APP_CFG["backtest"]["initial_balance"]),
        fee_bps=float(APP_CFG["backtest"]["fee_bps"]),
        slippage_bps=float(APP_CFG["backtest"]["slippage_bps"]),
    )

    model_total, model_sharpe, model_mdd = summarize_backtest(model_equity, model_returns)
    expert_total, expert_sharpe, expert_mdd = summarize_backtest(expert_equity, expert_returns)

    mode_results.append({
        "mode": mode_name,
        "policy": "position_cloning_rf",
        "total_return_pct": model_total,
        "sharpe": model_sharpe,
        "max_drawdown_pct": model_mdd,
    })
    mode_results.append({
        "mode": mode_name,
        "policy": "oracle_expert",
        "total_return_pct": expert_total,
        "sharpe": expert_sharpe,
        "max_drawdown_pct": expert_mdd,
    })

    beat_summary_df, beat_detail_df = summarize_symbol_outperformance(
        model_stateful_df,
        policy_name="position_cloning_rf",
        mode_name=mode_name,
    )
    beat_summary_frames.append(beat_summary_df)
    beat_detail_frames.append(beat_detail_df)
    oracle_beat_summary_df, oracle_beat_detail_df = summarize_symbol_outperformance(
        expert_stateful_df,
        policy_name="oracle_expert",
        mode_name=mode_name,
    )
    beat_summary_frames.append(oracle_beat_summary_df)
    beat_detail_frames.append(oracle_beat_detail_df)

    mode_stateful_frames[mode_name] = model_stateful_df
    mode_equity_series[f"position_cloning_rf_{mode_name}"] = model_equity
    mode_equity_series[f"oracle_expert_{mode_name}"] = expert_equity

buy_hold_stateful_df, bh_equity, bh_returns = backtest_position_policy_equal_weight(
    buy_hold_backtest_df,
    position_col="policy_position_label",
    initial_balance=float(APP_CFG["backtest"]["initial_balance"]),
    fee_bps=float(APP_CFG["backtest"]["fee_bps"]),
    slippage_bps=float(APP_CFG["backtest"]["slippage_bps"]),
)
bh_total, bh_sharpe, bh_mdd = summarize_backtest(bh_equity, bh_returns)
mode_results.append({
    "mode": "benchmark",
    "policy": "buy_and_hold_equal_weight",
    "total_return_pct": bh_total,
    "sharpe": bh_sharpe,
    "max_drawdown_pct": bh_mdd,
})

backtest_summary_df = pd.DataFrame(mode_results)
strategy_symbol_beat_summary_df = pd.concat(beat_summary_frames, ignore_index=True)
strategy_symbol_beat_detail_df = pd.concat(beat_detail_frames, ignore_index=True)

display(backtest_summary_df)
display(strategy_symbol_beat_summary_df)

position_cloning_preview_df = pd.concat(
    [
        mode_stateful_frames[mode_name].assign(mode=mode_name)
        for mode_name in ["long_only", "short_only", "long_short"]
    ],
    ignore_index=True,
)
display(position_cloning_preview_df[[col for col in [
    "mode",
    "symbol",
    "date_text",
    "pred_position_label",
    "mode_position_label",
    "prob_flat",
    "prob_long",
    "prob_short",
    "sim_position_before",
    "pred_action",
    "sim_position_after",
    "sim_position_label_after",
] if col in position_cloning_preview_df.columns]].head(120))

policy_equity_df = pd.DataFrame({"date": bh_equity.index, "buy_and_hold_equal_weight": bh_equity.values})
for series_name, equity_s in mode_equity_series.items():
    policy_equity_df[series_name] = equity_s.reindex(policy_equity_df["date"]).values
policy_equity_df = policy_equity_df.dropna()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(policy_equity_df["date"], policy_equity_df["position_cloning_rf_long_only"], label="RF Long Only", linewidth=2.0)
ax.plot(policy_equity_df["date"], policy_equity_df["position_cloning_rf_short_only"], label="RF Short Only", linewidth=2.0)
ax.plot(policy_equity_df["date"], policy_equity_df["position_cloning_rf_long_short"], label="RF Long + Short", linewidth=2.0)
ax.plot(policy_equity_df["date"], policy_equity_df["buy_and_hold_equal_weight"], label="Buy & Hold", linewidth=2.0)
ax.set_title("Oracle Imitation Position Cloning by Exposure Mode")
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
